[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/geometry/sundial/sundial.ipynb)

# Sundials

This example computes the hour lines and the seasons' shadow paths of a horizontal sundial. A sundial tells the time by a shadow. Its gnomon has an edge, the style, that points at the celestial pole, the spot in the sky the earth's axis points to; the sky, and the sun with it, turns once a day about that axis, and the style's shadow turns with it, along a fixed line of the dial for each hour. The tip of the gnomon, the nodus, casts a point whose path over a day tells the season.

A direction to the sun is modelled as an antivector, two rotor turns of the celestial equator's highest point: towards the pole by the declination, then about the pole by the hour angle. A point's height above the dial is its pairing with the dial plane, `dial & point`, so the shadow of the nodus lies where that pairing vanishes along the ray from the sun. Over a day the rays keep their angle to the pole and sweep a cone, a quadric of rays: a map from each ray to the plane that reads it, and the one place the metric enters. The cone's cut with the dial is the season's shadow path, a conic, and the summer and winter days as far from the equinox are the two branches of one hyperbola.

In the notation of spherical astronomy the sun's altitude reads as $\sin h = \sin\varphi \sin\delta + \cos\varphi \cos\delta \cos H$, at latitude $\varphi$, declination $\delta$ and hour angle $H$, and in the notation of gnomonics a horizontal dial's hour line makes the angle $\arctan(\sin\varphi \tan H)$ with the noon line.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext
from numga.algebras import VGA3D as ga
from examples.util.animation import save_figure, show_animation
from examples.geometry.sundial import render

mv = NumpyContext(ga).multivector
# Directions to the sun, the pole, and the compass points, and points on the dial are antivectors,
# and planes through the origin are vectors.
Point = ga.gatype.antivector()                      # Antivector-
Plane = ga.gatype.vector()                          # Vector+

# The local frame at the dial, and the horizontal dial plane through the origin.
east, north, up = mv.basis().dual()                 # [] Point-, Point-, Point-
west = -east                                        # [] Point-
dial = mv.z                                         # [] Plane+
# Solar noon, and the sun's hour angle per hour, a twenty-fourth of a turn.
noon = 12
turn_per_hour = 2 * np.pi / 24

## 1. The sun's two turns

The celestial pole is north raised towards the zenith by the latitude, and the celestial equator's highest point lies a quarter turn further; the sun stands there at noon on the equinoxes. Any other sun is that point turned twice: towards the pole by the declination, the season's angle north or south of the equator, then about the pole towards the west by the hour angle, a twenty-fourth of a turn per hour. At noon the sun stands a quarter turn less the latitude, plus the declination, above the horizon.

In [ ]:
# The dial's latitude and the tilt of the earth's axis, the sun's declination at the solstices.
latitude = np.radians(52.0)
tilt = np.radians(23.44)
solstices_and_equinox = np.array([-tilt, 0.0, tilt])

In [ ]:
def sky(latitude: float) -> tuple[Point, Point]:
    """The celestial pole is north raised towards the zenith by the latitude, and the celestial equator's highest point lies a quarter turn further."""
    # The commutator of two orthogonal unit directions is the bivector of the meridian plane they span.
    meridian = up.commutator(north)                                            # [] Bivector+
    pole_turn = (meridian * (-latitude / 2)).exp()                             # [] Rotor+
    equator_turn = (meridian * (-(latitude + np.pi / 2) / 2)).exp()            # [] Rotor+
    return pole_turn >> north, equator_turn >> north


def sun(pole: Point, equator: Point, declinations: np.ndarray, hour_angles: np.ndarray) -> Point:
    """The direction to the sun is the equator's highest point turned towards the pole by the declination and about the pole towards the west by the hour angle."""
    # The declination turns in the meridian plane, the hour angle in the plane of the celestial equator.
    declination_turns = (pole.commutator(equator) * (-declinations / 2)).exp() # [...] Rotor+
    hour_turns = (west.commutator(equator) * (-hour_angles / 2)).exp()         # [...] Rotor+
    return (hour_turns * declination_turns) >> equator                         # [...] Point-


pole, equator = sky(latitude)                                                  # [] Point-, Point-
noon_suns = sun(pole, equator, solstices_and_equinox, np.zeros_like(solstices_and_equinox))             # [seasons] Point-
print("noon heights, winter, equinox, summer:", np.degrees(np.arcsin((dial & noon_suns).to_array())).round(2))

## 2. The hour lines

The nodus casts its shadow where the ray from the sun past the nodus meets the dial. The style runs through the nodus towards the pole, and the plane through the style and the sun at a given hour holds the sun at that hour on every day of the year: the declination turns the sun within that plane. So each hour's shadow, whatever the season, falls along one line of the dial through the foot of the style. Measured from the noon line, the hour lines crowd towards noon and spread towards morning and evening.

In [ ]:
# The height of the nodus above the dial, the hours drawn with their hour angles, and the declinations
# sampled from solstice to solstice.
height = 1.0
hours = np.arange(6, 19)
hour_angles = (hours - noon) * turn_per_hour
n_declinations = 25

In [ ]:
def shadow(nodus: Point, suns: Point) -> Point:
    """The shadow of the nodus lies on the ray from each sun past the nodus, where its height above the dial vanishes."""
    # Each point's height above the horizontal dial is its pairing with the dial plane.
    return nodus - suns * ((dial & nodus) / (dial & suns))                     # [...] Point-


nodus = up * height                                                            # [] Point-
# Where the style meets the dial.
foot = shadow(nodus, pole)                                                     # [] Point-
declinations = np.linspace(-tilt, tilt, n_declinations)                        # [declinations]
hour_suns = sun(pole, equator, declinations[None, :], hour_angles[:, None])    # [hours, declinations] Point-
hour_lines = shadow(nodus, hour_suns)                                          # [hours, declinations] Point-
offsets = hour_lines - foot                                                    # [hours, declinations] Point-
# The bearing of each offset across the dial from north towards east reads from its pairings with x = 0 and y = 0.
bearings = np.arctan2((mv.x & offsets).to_array(), (mv.y & offsets).to_array())
print("hour line bearings from the noon line, in degrees:", np.degrees(bearings[:, -1]).round(1))

## 3. The seasons as conics

On a given day the ray from the nodus to the sun keeps its angle to the pole, a quarter turn less the declination, so the day's rays sweep a cone about the pole. A cone is a quadric of rays, a map from each ray to the plane that reads it, and a ray lies on it where that reading of itself, `cones(ray) & ray`, is zero. The metric enters in the cone alone, where a ray's height along the pole is compared with its length. The dial cuts the cone, and the shadow path is the cut, a conic. The form only sees the declination squared, so its other nappe holds the rays of the opposite declination: a summer day and the winter day as far from the equinox are the two branches of one hyperbola, drawn here in one colour. At the equinoxes the cone flattens into the plane of the celestial equator, and the shadow runs along a straight line.

In [ ]:
# The seasons drawn, a whole number of months from an equinox up to a solstice on either side, and the
# hours of a day sampled with their hour angles.
n_months = 3
day_hours = np.arange(3, 21.01, 0.05)
day_angles = (day_hours - noon) * turn_per_hour

In [ ]:
seasons = tilt * np.sin(np.pi / 2 * np.arange(-n_months, n_months + 1) / n_months)  # [seasons]
days = sun(pole, equator, seasons[:, None], day_angles[None, :])               # [seasons, samples] Point-
rays = (nodus - shadow(nodus, days)).normalized()                              # [seasons, samples] Point-
# The plane across the pole reads a ray's height along it: for a unit sun, the sine of its declination.
across = pole.dual_inverse()                                                   # [] Plane+
# A ray lies on its season's cone when its height squared equals the sun's times its own length squared.
cones = across * (across & Point) - (across & days[:, 0]).squared() * Point.dual_inverse()  # [seasons] Plane+ <- Point-
save_figure(render.draw_dial(nodus, foot, seasons, days, hour_suns, hours), "sundial");


## 4. A year

Through the year the declination swings between the solstices, and the cone of the day's rays opens and closes about the pole: in summer it stands steep and cuts the dial in a tight hyperbola near the gnomon, at the equinoxes it flattens into a plane and traces a straight line, and in winter it lies shallow along the hyperbola's far branch. In the animation, the left panel traces the day's rays past the nodus onto the dial in space, and the right panel draws that day's shadow path across the dial face.

In [ ]:
# The frames of one year, the hours a frame's rays are drawn at with their hour angles, and the frame duration.
n_frames = 108
ray_hours = np.arange(4, 21, 0.25)
ray_angles = (ray_hours - noon) * turn_per_hour
duration_ms = 40

In [ ]:
# From solstice to solstice; the animation plays the year forward and back.
year_declinations = tilt * np.sin(np.linspace(-np.pi / 2, np.pi / 2, n_frames // 2 + 1))  # [frames]
year_rays = sun(pole, equator, year_declinations[:, None], ray_angles[None, :])           # [frames, ray_hours] Point-
year_days = sun(pole, equator, year_declinations[:, None], day_angles[None, :])           # [frames, samples] Point-

show_animation(render.animate_year(year_declinations, year_rays, year_days, nodus, foot, seasons, days, hour_suns, hours),
               "sundial_year", duration_ms, forward_then_back=True)

In [ ]:
# checks
# The noon sun stands a quarter turn less the latitude, plus the declination, above the horizon.
assert np.abs(np.arcsin((dial & noon_suns).to_array()) - (np.pi / 2 - latitude + solstices_and_equinox)).max() < 1e-10
# Every shadow ray of a season's day lies on its cone, and each hour line lies at the angle of gnomonics.
assert np.abs((cones[:, None](rays) & rays).to_array()).max() < 1e-10
expected = np.arctan2(np.sin(latitude) * np.sin(hour_angles), np.cos(hour_angles))
assert np.abs(np.sin(bearings - expected[:, None])).max() < 1e-10